# Análisis de Cobertura Indoor en Edificio Multiplanta

## Campus universitario: Evaluación de Penetración de Señal y Necesidad de Refuerzo

**Objetivo:** Determinar la cobertura celular en un edificio de varias plantas y proponer soluciones de mejora si es necesario.

**Escenario:** Edificio de 4 plantas + semiótano técnico. Se evalúan 3 puntos de usuario para servicios de voz, mensajería y acceso a datos.

**Conceptos clave:**
- Pérdida de propagación exterior (espacio libre)
- Pérdida de penetración por fachada, paredes interiores y forjados
- Presupuesto de enlace (link budget)
- Comparación de soluciones de mejora (DAS, repetidor, small cell)

**Frecuencias a estudiar:** LTE 1800 MHz (principal), UMTS 2100 MHz (respaldo), 700/800 MHz (cobertura), 2600 MHz (capacidad)



## 1. Importar Librerías y Configurar Entorno


In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Rectangle
import seaborn as sns
from typing import Dict, List, Tuple
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 10

# Añadir ruta del proyecto para importar módulos
sys.path.insert(0, '/workspaces/Cobertura-indoor-de-campus')

# Importar módulos del proyecto
from src.propagation import (Frequency, PropagationModel, ServiceThreshold, 
                              IndoorMeasurementPoint)
from src.visualization import BuildingVisualization, ReportGenerator
from data.building_model import *

print("✓ Librerías importadas correctamente")
print(f"✓ Proyecto: {BUILDING_MODEL['name']}")
print(f"✓ Versión del notebook: 1.0")


## 2. Definir Parámetros de Propagación y Umbrales de Servicio

Se establecen los parámetros de las portadoras celulares y los umbrales mínimos de recepción para que funcionen correctamente los diferentes servicios.

**Parámetros de antena exterior:**
- Altura: 25 metros (típico de macro celda)
- Potencia: 43 dBm (20 W)
- Ganancia: 17 dBi
- Distancia al edificio: ~45 metros (fachada más cercana)

**Pérdidas constructivas típicas:**
- Fachada (hormigón): 12-18 dB
- Pared interior (ladrillo): 3-5 dB
- Forjado (hormigón): 12-18 dB


In [ ]:
# definir portadoras de prueba
frequencies = {
    'LTE_1800': Frequency(
        name='LTE 1800 MHz',
        frequency_mhz=1800,
        technology='LTE',
        bandwidth_mhz=20,
        power_dbm=43
    ),
    'UMTS_2100': Frequency(
        name='UMTS 2100 MHz',
        frequency_mhz=2100,
        technology='UMTS',
        bandwidth_mhz=5,
        power_dbm=43
    ),
    'LTE_800': Frequency(
        name='LTE 800 MHz',
        frequency_mhz=800,
        technology='LTE',
        bandwidth_mhz=10,
        power_dbm=43
    ),
    'LTE_700': Frequency(
        name='LTE 700 MHz',
        frequency_mhz=700,
        technology='LTE',
        bandwidth_mhz=10,
        power_dbm=43
    ),
    '5G_2600': Frequency(
        name='5G NR 2600 MHz',
        frequency_mhz=2600,
        technology='5G NR',
        bandwidth_mhz=100,
        power_dbm=40
    )
}

# parámetros de antena exterior (macro celda)
antena = {
    'lte_1800_main': {
        'power_dbm': 43,  # 20W
        'gain_dbi': 17,
        'height_m': 25,
        'position': (-15, -10),  # Fuera del edificio
        'eirp_dbm': 43 + 17  # Potencia radiada isotrópica equivalente
    }
}

# Umbrales de servicio (en dBm)
service_thresholds = {
    'voz': -100,              # VoLTE necesita al menos -100 dBm
    'sms': -105,              # SMS es más robusto
    'datos_basicos': -95,     # Notificaciones, mapas
    'videoconf': -85,         # Videoconferencia, streaming
    'descarga': -75           # Descarga/upload rápido
}

# Mostrar información
print("=" * 70)
print("PARÁMETROS DE PORTADORA Y SERVICIO")
print("=" * 70)
for key, freq in frequencies.items():
    print(f"\n{freq.name}:")
    print(f"  Frecuencia: {freq.frequency_mhz} MHz")
    print(f"  Tecnología: {freq.technology}")
    print(f"  Longitud de onda: {freq.wavelength_m:.4f} m")
    print(f"  Coeficiente pérdida: {freq.path_loss_coefficient:.2f} dB")

print("\n" + "=" * 70)
print("UMBRALES DE SERVICIO (dBm)")
print("=" * 70)
for service, threshold in service_thresholds.items():
    print(f"  {service.upper()}: {threshold} dBm")



## 3. Calcular Pérdidas de Propagación Exterior

Utilizamos el modelo de espacio libre (Friis) para calcular la atenuación desde la antena exterior hasta los puntos de Usuario. La fórmula es:

$$L_{fs} = 20 \log_{10}(f [MHz]) + 20 \log_{10}(d [m]) + 20 \log_{10}(4\pi / c)$$

donde $c = 3 \times 10^8$ m/s es la velocidad de la luz.


In [ ]:
# Cargar edificio y puntos de medida del módulo de datos
building = get_building()
measurement_points_data = get_measurement_points()

# Calcular distancias exteriores para cada punto
print("=" * 70)
print("CÁLCULO DE PÉRDIDAS DE PROPAGACIÓN EXTERIOR")
print("=" * 70)

# Posición de antena exterior
tx_pos = (building['antenna']['position_x_m'], 
          building['antenna']['position_y_m'])
print(f"\nPosición antena exterior: ({tx_pos[0]}, {tx_pos[1]}) metros")
print(f"Altura: {building['antenna']['height_m']} metros")
print(f"Potencia transmitida: {building['antenna']['power_dbm']} dBm")
print(f"Ganancia antena: {building['antenna']['gain_dbi']} dBi")

# Calcular pérdidas para cada punto
exterior_losses = {}
for point_data in measurement_points_data:
    # Distancia 2D en el plano
    dx = point_data['x_m'] - tx_pos[0]
    dy = point_data['y_m'] - tx_pos[1]
    distance_2d = np.sqrt(dx**2 + dy**2)
    
    # Distancia 3D incluyendo altura
    dz = point_data['z_m'] - building['antenna']['height_m']
    distance_3d = np.sqrt(distance_2d**2 + dz**2)
    
    exterior_losses[point_data['name']] = {
        'distance_2d': distance_2d,
        'distance_3d': distance_3d,
        'dz': dz,
        'losses_1800': PropagationModel.free_space_loss(frequencies['LTE_1800'], distance_3d),
        'losses_2100': PropagationModel.free_space_loss(frequencies['UMTS_2100'], distance_3d),
        'losses_700': PropagationModel.free_space_loss(frequencies['LTE_700'], distance_3d),
    }
    
# Mostrar tabla de pérdidas
exterior_df = pd.DataFrame({
    'Punto de Medida': [key for key in exterior_losses.keys()],
    'Distancia 2D (m)': [exterior_losses[k]['distance_2d'] for k in exterior_losses.keys()],
    'Distancia 3D (m)': [exterior_losses[k]['distance_3d'] for k in exterior_losses.keys()],
    'Pérdida 1800 MHz (dB)': [exterior_losses[k]['losses_1800'] for k in exterior_losses.keys()],
    'Pérdida 2100 MHz (dB)': [exterior_losses[k]['losses_2100'] for k in exterior_losses.keys()],
    'Pérdida 700 MHz (dB)': [exterior_losses[k]['losses_700'] for k in exterior_losses.keys()],
})

print("\n")
print(exterior_df.to_string(index=False))



## 4. Modelar Penetración Interior por Elementos Constructivos

Las pérdidas de penetración se acumulan según el trayecto del rayo de radio:

- **Entrada principal:** Solo atraviesa fachada (1 elemento) → ej. 15 dB
- **Aula 2ª planta:** Fachada + paredes interiores + 2 forjados (4 elementos) → ej. 15 + 4 + 15 + 15 = 49 dB
- **Semiótano técnico:** Fachada + paredes + 1 forjado (3 elementos) → ej. 15 + 4 + 15 = 34 dB


In [ ]:
# Crear tabla con pérdidas de penetración
print("\n" + "=" * 70)
print("PÉRDIDAS DE PENETRACIÓN POR ELEMENTOS CONSTRUCTIVOS")
print("=" * 70)

penetration_df = ReportGenerator.create_loss_table(
    # Convertir datos a objetos IndoorMeasurementPoint
    [IndoorMeasurementPoint(
        name=p['name'],
        location=p['location'],
        x_m=p['x_m'],
        y_m=p['y_m'],
        z_m=p['z_m'],
        distance_outdoor_m=exterior_losses[p['name']]['distance_3d'],
        penetration_losses=p['penetration_losses']
    ) for p in measurement_points_data]
)

print("\n")
print(penetration_df.to_string(index=False))

# Guardar penetration_df para usar luego
penetration_losses_summary = {p['name']: p['penetration_losses'] 
                              for p in measurement_points_data}



## 5. Evaluar Cobertura en Tres Puntos de Usuario

Calcularemos el nivel de señal recibida interior (en dBm) para cada punto:

$$P_{rx,interior} = P_{tx} + G_{tx} - L_{propagacion} - L_{penetracion}$$

donde:
- $P_{tx}$ = Potencia transmitida (43 dBm)
- $G_{tx}$ = Ganancia de antena (17 dBi)
- $L_{propagacion}$ = Pérdida en espacio libre (calculada)
- $L_{penetracion}$ = Suma de pérdidas de elementos constructivos

Después compararemos con umbrales de servicio.


In [ ]:
# Calcular niveles de recepción con LTE 1800 MHz (principal)
print("\n" + "=" * 70)
print("EVALUACIÓN DE COBERTURA - LTE 1800 MHz")
print("=" * 70)

freq_principal = frequencies['LTE_1800']
tx_power_dbm = antena['lte_1800_main']['power_dbm']
tx_gain_dbi = antena['lte_1800_main']['gain_dbi']
eirp_dbm = tx_power_dbm + tx_gain_dbi

measurement_results = []

for point_data in measurement_points_data:
    point_name = point_data['name']
    
    # Pérdida exterior
    loss_exterior = exterior_losses[point_name]['losses_1800']
    
    # Pérdida de penetración
    penetration_loss = sum(penetration_losses_summary[point_name].values())
    
    # Pérdida total
    total_loss = loss_exterior + penetration_loss
    
    # Nivel recibido
    rx_level_dbm = eirp_dbm - total_loss
    
    # Evaluar servicios
    voz_ok = rx_level_dbm >= service_thresholds['voz']
    datos_ok = rx_level_dbm >= service_thresholds['datos_basicos']
    
    measurement_results.append({
        'Punto': point_name,
        'Distancia (m)': round(exterior_losses[point_name]['distance_3d'], 1),
        'Pérdida Exterior (dB)': round(loss_exterior, 1),
        'Pérdida Penetración (dB)': round(penetration_loss, 1),
        'Pérdida Total (dB)': round(total_loss, 1),
        'Nivel Recibido (dBm)': round(rx_level_dbm, 1),
        'Margen para Voz (dB)': round(rx_level_dbm - service_thresholds['voz'], 1),
        'Margen para Datos (dB)': round(rx_level_dbm - service_thresholds['datos_basicos'], 1),
        'Voz': '✓ SÍ' if voz_ok else '✗ NO',
        'Datos': '✓ SÍ' if datos_ok else '✗ NO'
    })

results_df = pd.DataFrame(measurement_results)
print("\n")
print(results_df.to_string(index=False))

print("\n" + "=" * 70)
print(f"EIRP (Potencia radiada isotrópica equiv.): {eirp_dbm} dBm")
print(f"Umbral servicio Voz: {service_thresholds['voz']} dBm")
print(f"Umbral servicio Datos: {service_thresholds['datos_basicos']} dBm")
print("=" * 70)

# Guardar para gráficos posteriores
results_by_point = {r['Punto']: r for r in measurement_results}



## 6. Visualizar Resultados con Mapas y Gráficos

Crearemos visualizaciones del edificio con los puntos de medida y un diagrama de niveles de señal recibida.


In [ ]:
# Crear puntos de medida para visualización
points_for_viz = [
    IndoorMeasurementPoint(
        name=p['name'],
        location=p['location'],
        x_m=p['x_m'],
        y_m=p['y_m'],
        z_m=p['z_m'],
        distance_outdoor_m=exterior_losses[p['name']]['distance_3d'],
        penetration_losses=p['penetration_losses']
    ) for p in measurement_points_data
]

# Visualizar plano del edificio
visualizer = BuildingVisualization(figsize=(14, 10))
tx_position = (building['antenna']['position_x_m'], 
               building['antenna']['position_y_m'])

fig, ax = visualizer.plot_building_layout(
    building_data=building,
    measurement_points=points_for_viz,
    tx_position=tx_position,
    title="PLANO DEL EDIFICIO CON PUNTOS DE MEDIDA\n(LTE 1800 MHz)"
)

# Añadir información de niveles de señal en anotación
info_text = "NIVELES DE RECEPCIÓN\n" + "=" * 25 + "\n"
for i, point in enumerate(points_for_viz):
    level = results_by_point[point.name]['Nivel Recibido (dBm)']
    voz_status = results_by_point[point.name]['Voz']
    info_text += f"{i+1}. {point.name}: {level} dBm ({voz_status})\n"

ax.text(0.02, 0.98, info_text, transform=ax.transAxes, 
        fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
        family='monospace')

plt.tight_layout()
plt.show()

print("\n✓ Plano del edificio visualizado")


In [ ]:
# Gráfico de comparación de servicios
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

services_to_plot = [
    ('voz', 'Voz (VoLTE)', service_thresholds['voz']),
    ('sms', 'SMS', service_thresholds['sms']),
    ('datos_basicos', 'Datos Básicos', service_thresholds['datos_basicos'])
]

for ax, (service_key, service_name, threshold) in zip(axes, services_to_plot):
    points = [r['Punto'] for r in measurement_results]
    levels = [r['Nivel Recibido (dBm)'] for r in measurement_results]
    margins = [level - threshold for level in levels]
    
    # Colores según margen
    colors = ['#2ecc71' if m >= 0 else '#e74c3c' for m in margins]
    
    bars = ax.barh(points, margins, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Umbral')
    ax.set_xlabel('Margen (dB)', fontweight='bold')
    ax.set_title(f'{service_name}\n(Umbral: {threshold} dBm)', fontweight='bold', fontsize=11)
    ax.grid(True, alpha=0.3, axis='x')
    
    # Añadir valores sobre barras
    for i, (bar, margin) in enumerate(zip(bars, margins)):
        ax.text(margin + 1, i, f'{margin:.1f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Gráficos de servicio por punto generados")



## 7. Proponer Soluciones de Mejora (DAS, Repetidor, Small Cell)

Para puntos con cobertura insuficiente, analizamos las opciones disponibles:

**Opción 1: DAS (Distributed Antenna System)**
- Mejora: 20-30 dB
- Costo: 100k-300k€
- Tiempo: 3-6 meses
- Aplica a: Edificios grandes con múltiples plantas

**Opción 2: Small Cell**
- Mejora: 15-20 dB
- Costo: 15k-30k€
- Tiempo: 2-4 semanas
- Aplica a: Zonas específicas críticas

**Opción 3: Repetidor**
- Mejora: 10-15 dB
- Costo: 3k-8k€
- Tiempo: 1-2 semanas
- Aplica a: Mejora puntual local


In [ ]:
# Análisis de necesidad de mejora
print("\n" + "=" * 70)
print("ANÁLISIS DE SOLUCIONES DE MEJORA")
print("=" * 70)

improvement_analysis = []

for result in measurement_results:
    point_name = result['Punto']
    current_level = result['Nivel Recibido (dBm)']
    voz_margin = result['Margen para Voz (dB)']
    datos_margin = result['Margen para Datos (dB)']
    
    # Determinar necesidad
    if voz_margin >= 5:
        status = "✓ BUENO - Sin mejora necesaria"
        recommendation = "Cobertura macro suficiente"
        gain_needed = 0
    elif voz_margin >= 0:
        status = "⚠ MARGINAL - Mejora recomendada"
        recommendation = "Repetidor (10 dB) o Small Cell (15 dB)"
        gain_needed = 10
    else:
        status = "✗ CRÍTICO - Mejora urgente"
        recommendation = "DAS (25 dB) o Small Cell (15-20 dB)"
        gain_needed = 15 - voz_margin
    
    # Calcular nivel con soluciones
    with_das = current_level + 25
    with_small_cell = current_level + 18
    with_repeater = current_level + 12
    
    improvement_analysis.append({
        'Punto': point_name,
        'Nivel Actual': current_level,
        'Con DAS': round(with_das, 1),
        'Con Small Cell': round(with_small_cell, 1),
        'Con Repetidor': round(with_repeater, 1),
        'Estado': status,
        'Recomendación': recommendation
    })

improvement_df = pd.DataFrame(improvement_analysis)

print("\n--- NIVELES PROYECTADOS CON SOLUCIONES ---\n")
print(improvement_df[['Punto', 'Nivel Actual', 'Con DAS', 'Con Small Cell', 'Con Repetidor']].to_string(index=False))

print("\n--- RECOMENDACIONES ---\n")
for i, row in improvement_df.iterrows():
    print(f"\n{i+1}. {row['Punto']}")
    print(f"   Estado: {row['Estado']}")
    print(f"   Recomendación: {row['Recomendación']}")



## 8. Análisis Comparativo de Frecuencias Alternativas

Comparamos cómo varía la cobertura si se utilizan otras bandas de frecuencia:

- **700 MHz:** Baja pérdida penetración, mejor cobertura indoor
- **800 MHz:** Similar a 1800 MHz pero con menos pérdida
- **1800 MHz:** Banda recomendada (referencia)
- **2100 MHz:** Banda de respaldo (UMTS)
- **2600 MHz:** 5G alto rendimiento pero muy atenuada

**Relación:** Pérdida ~= 20·log₁₀(f₂/f₁), por lo que:
- 1800 vs 700 MHz → +7 dB más pérdida cada 10 km
- 2600 vs 1800 MHz → +3.3 dB más pérdida


In [ ]:
# Calcular cobertura con diferentes frecuencias
print("\n" + "=" * 70)
print("ANÁLISIS COMPARATIVO DE FRECUENCIAS")
print("=" * 70)

comparison_data = []

# Frecuencias a comparar
freq_list = ['LTE_700', 'LTE_800', 'LTE_1800', 'UMTS_2100', '5G_2600']

for point_data in measurement_points_data:
    point_name = point_data['name']
    row_data = {'Punto': point_name}
    
    for freq_key in freq_list:
        freq = frequencies[freq_key]
        
        # Pérdida exterior
        loss_exterior = PropagationModel.free_space_loss(freq, 
                                                        exterior_losses[point_name]['distance_3d'])
        
        # Pérdida penetración (igual para todas las frecuencias, asumiendo el mismo punto)
        loss_penetration = sum(penetration_losses_summary[point_name].values())
        
        # Nivel recibido (con 43 dBm de EIRP)
        rx_level = antena['lte_1800_main']['eirp_dbm'] - loss_exterior - loss_penetration
        
        # Comprobar voz
        voz_ok = '✓' if rx_level >= service_thresholds['voz'] else '✗'
        
        freq_label = f"{freq.frequency_mhz:.0f}MHz\n({rx_level:.0f}dBm {voz_ok})"
        row_data[freq.frequency_mhz] = {
            'rx_level': rx_level,
            'label': freq_label,
            'voz_ok': voz_ok
        }
    
    comparison_data.append(row_data)

# Crear tabla de comparativa
print("\n--- NIVEL DE RECEPCIÓN POR FRECUENCIA (dBm) ---\n")
comparison_summary = {
    'Punto': [d['Punto'] for d in comparison_data],
    '700 MHz': [d[700]['rx_level'] for d in comparison_data],
    '800 MHz': [d[800]['rx_level'] for d in comparison_data],
    '1800 MHz': [d[1800]['rx_level'] for d in comparison_data],
    '2100 MHz': [d[2100]['rx_level'] for d in comparison_data],
    '2600 MHz': [d[26000]['rx_level'] for d in comparison_data],
}

comparison_table = pd.DataFrame(comparison_summary)
comparison_table.iloc[:, 1:] = comparison_table.iloc[:, 1:].round(1)

print(comparison_table.to_string(index=False))

# Gráfico comparativo
fig, ax = plt.subplots(figsize=(14, 6))

freq_names = ['700 MHz', '800 MHz', '1800 MHz', '2100 MHz', '2600 MHz']
x = np.arange(len(measurement_points_data))
width = 0.16

colors = ['#2c3e50', '#34495e', '#7f8c8d', '#95a5a6', '#bdc3c7']

for i, freq_key in enumerate(freq_list):
    levels = [comparison_data[j][frequencies[freq_key].frequency_mhz]['rx_level'] 
              for j in range(len(comparison_data))]
    ax.bar(x + i * width, levels, width, label=freq_names[i], 
           color=colors[i], alpha=0.8, edgecolor='black', linewidth=0.5)

# Línea de umbral voz
ax.axhline(y=service_thresholds['voz'], color='red', linestyle='--', 
          linewidth=2, label=f"Umbral Voz ({service_thresholds['voz']} dBm)")

ax.set_ylabel('Nivel de Recepción (dBm)', fontweight='bold', fontsize=11)
ax.set_title('COMPARATIVA DE COBERTURA POR FRECUENCIA', fontweight='bold', fontsize=12)
ax.set_xticks(x + width * 2)
ax.set_xticklabels([d['Punto'] for d in comparison_data])
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(-120, -50)

plt.tight_layout()
plt.show()

print("\n✓ Gráfico comparativo generado")



## 9. Conclusiones y Recomendación Final

### Entregables del Ejercicio:

1. **✓ Tabla de pérdidas por tramo** - Generada anteriormente
2. **✓ Croquis del edificio** - Con puntos de medida y antena
3. **✓ Evaluación de servicios** - Comparativa voz/datos para cada punto
4. **✓ Análisis de soluciones** - DAS, Small Cell, Repetidor
5. **✓ Comparativa de frecuencias** - 700 MHz a 2600 MHz


In [ ]:
# Resumen ejecutivo final
print("\n\n")
print("╔" + "=" * 68 + "╗")
print("║" + "RESUMEN EJECUTIVO - ANÁLISIS DE COBERTURA INDOOR".center(68) + "║")
print("╚" + "=" * 68 + "╝")

print(f"\nFRECUENCIA PRINCIPAL: LTE 1800 MHz")
print(f"POTENCIA TRANSMITIDA: {antena['lte_1800_main']['power_dbm']} dBm")
print(f"ALTURA ANTENA: {building['antenna']['height_m']} metros")
print(f"UBICACIÓN: Exterior, a ~{building['reference_distances']['to_facade']}m de fachada\n")

# Resumen por punto
critical = 0
weak = 0
good = 0

print("┌─ COBERTURA POR PUNTO ─────────────────────────────────────────┐")
for result in measurement_results:
    level = result['Nivel Recibido (dBm)']
    voz = result['Voz']
    
    if voz == '✓ SÍ':
        icon = "✓"
        status = "BUENA"
        good += 1
    elif level < -100:
        icon = "✗"
        status = "CRÍTICA"
        critical += 1
    else:
        icon = "⚠"
        status = "DÉBIL"
        weak += 1
    
    print(f"│ {icon} {result['Punto']:25} {level:7.1f} dBm  {status:8}  {voz}  │")

print("└───────────────────────────────────────────────────────────────┘")

print(f"\n┌─ RESUMEN ESTADÍSTICO ──────────────────────────────────────────┐")
print(f"│ Puntos con cobertura BUENA:    {good} ({'✓' if good == 3 else '⚠'})                              │")
print(f"│ Puntos con cobertura DÉBIL:    {weak}                                  │")
print(f"│ Puntos con cobertura CRÍTICA:  {critical}                                  │")
print(f"└───────────────────────────────────────────────────────────────┘")

# Conclusión
print("\n┌─ CONCLUSIÓN Y DECISIÓN ────────────────────────────────────────┐")

if good == 3:
    conclusion = """
│ ✓ COBERTURA SUFICIENTE: La macro celda exterior proporciona   │
│   cobertura adecuada en los tres puntos estudiados.           │
│                                                               │
│ DECISIÓN: No se requiere refuerzo indoor (DAS, Small Cell    │
│           o Repetidor) para los servicios básicos.           │
│                                                               │
│ NOTA: Monitorear nivel en semiótano para videollamadas.      │
"""
elif good >= 1 and weak > 0:
    conclusion = """
│ ⚠ COBERTURA PARCIAL: Se detectan zonas débiles o críticas.   │
│                                                               │
│ DECISIÓN: Considerar Small Cell o Repetidor en:             │
│           • Semiótano técnico (prioridad)                    │
│           • Posibles refuerzos adicionales en aulas           │
│                                                               │
│ PRESUPUESTO ESTIMADO: 15k-30k€ (Small Cell)                 │
│ PLAZO: 2-4 semanas                                           │
"""
else:
    conclusion = """
│ ✗ COBERTURA INSUFICIENTE: Múltiples puntos sin servicio.     │
│                                                               │
│ DECISIÓN: Implementar DAS o múltiples Small Cells            │
│                                                               │
│ PRESUPUESTO ESTIMADO: 100k-300k€ (DAS) o 30k-60k€ (SC)     │
│ PLAZO: 3-6 meses (DAS) o 1-2 meses (Small Cells)            │
"""

print(conclusion)
print("└───────────────────────────────────────────────────────────────┘\n")

print("FIN DEL ANÁLISIS")
